## **집계연산**

### **PySpark 설정**

In [1]:
# [+] pyspark 불러오기
from pyspark import SparkConf, SparkContext
import pandas as pd

In [2]:
# [+] SparkConf, SparkContext 객체 생성하기
conf = SparkConf().setMaster('local').setAppName('A')

In [3]:
sc = SparkContext(conf=conf)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 04:05:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### **reduce( ) 메서드**

In [4]:
# [+] x+y를 수행하는 reduce() 
sc.parallelize([1, 2, 3, 4, 5]).reduce(lambda x,y : x+y)

15

In [5]:
# reduce() 는 각 파티션별로 수행됨
# [+] 파티션 수: 2
rdd = sc.parallelize([1,2,3,4,5], 2)
rdd.collect()

[1, 2, 3, 4, 5]

In [6]:
# [+] glom() : 파티션 별 RDD 값들을 배열에 모으는 변환 메서드
rdd.glom().collect()

[[1, 2], [3, 4, 5]]

In [7]:
# [+] 파티션 2개에 대해 x+y를 수행하는 reduce()
rdd.reduce(lambda x,y : x+y)

15

In [8]:
# [+] reduce() 주의점: 파티션 구성에 따라 결과 값이 달라질 수 있음
# 예: lambda x, y: (x * 2) + y
sc.parallelize([1,2,3,4,5],2).reduce(lambda x,y : (x ** 2) + y)

183

In [9]:
# 파티션 수: 2
sc.parallelize([1,2,3,4,5], 2).reduce(lambda x,y: (x * 2) + y)

33

In [10]:
sc.parallelize([1,2,3,4,5],3).reduce(lambda x,y : (x ** 2) + y)

85

### **10000 개 정수 난수의 총합 계산하기**

In [11]:
!pip install numpy

In [12]:
# [+] numpy 라이브러리 불러오기
import numpy as np

In [13]:
# [+] [0, 100] 범위의 정수 난수 10000개 생성
x = np.random.randint(0, 100, size=10000)
x[:100]

array([63, 44, 10, 70, 52, 52, 13, 60, 75,  0, 25, 46, 34, 47, 30, 11, 18,
       75, 83, 33, 93, 57, 66, 38, 39, 37, 99,  5, 58, 45, 81, 75, 86, 23,
       51, 29, 49,  4, 82, 85, 86, 26, 97, 96, 13, 17, 97, 78, 30, 93, 82,
       89, 99, 34, 57, 40, 68, 80, 88, 22, 21, 10, 42, 66,  5, 52, 85, 49,
       55, 33, 32, 87,  7, 61, 38, 94, 69,  8,  5, 75,  1, 26, 97, 88, 59,
       33, 89, 69, 35, 26, 44, 69, 62, 45, 72, 63, 51, 30, 84, 19])

In [14]:
# [+] 생성한 난수 배열을 RDD로 변환(파티션 수: 8개)
rdd = sc.parallelize(x,8)

In [15]:
# 파티션 별 RDD 값 확인
partitioned_data = rdd.glom().collect()
for i, partition in enumerate(partitioned_data):
    print(f"Partition {i}: {partition}\n")

[Stage 7:============================================>              (6 + 1) / 8]

Partition 0: [np.int64(63), np.int64(44), np.int64(10), np.int64(70), np.int64(52), np.int64(52), np.int64(13), np.int64(60), np.int64(75), np.int64(0), np.int64(25), np.int64(46), np.int64(34), np.int64(47), np.int64(30), np.int64(11), np.int64(18), np.int64(75), np.int64(83), np.int64(33), np.int64(93), np.int64(57), np.int64(66), np.int64(38), np.int64(39), np.int64(37), np.int64(99), np.int64(5), np.int64(58), np.int64(45), np.int64(81), np.int64(75), np.int64(86), np.int64(23), np.int64(51), np.int64(29), np.int64(49), np.int64(4), np.int64(82), np.int64(85), np.int64(86), np.int64(26), np.int64(97), np.int64(96), np.int64(13), np.int64(17), np.int64(97), np.int64(78), np.int64(30), np.int64(93), np.int64(82), np.int64(89), np.int64(99), np.int64(34), np.int64(57), np.int64(40), np.int64(68), np.int64(80), np.int64(88), np.int64(22), np.int64(21), np.int64(10), np.int64(42), np.int64(66), np.int64(5), np.int64(52), np.int64(85), np.int64(49), np.int64(55), np.int64(33), np.int64(3

In [16]:
# [+] x+y 집계 연산
rdd.reduce(lambda x,y : x+y)

np.int64(490564)

### **fold( ) 메서드**
```reduce()```와 유사하며, 초기 값을 지정하여 집계를 수행

In [17]:
# [+] 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(1, lambda x,y : x+y)

12

In [18]:
# [+] 파티션 수가 2개이며, 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1,2,3,4],2).fold(1, lambda x,y : x+y)

13

In [19]:
# [+] 초기 값이 0이 주어지고, x*y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(0, lambda x,y : x*y)

0

### **groupBy( ) 메서드**
```groupBy()```: 인자 함수를 기준으로 그룹핑을 수행


In [35]:
# [+] 홀수/짝수로 그룹핑을 수행하는 groupBy() 코드
res = sc.parallelize(np.random.randint(1,10, size=8)).groupBy(lambda x : x % 2).collect()

In [36]:
# [+] groupBy() 결과 출력하기
for k,v in res:
    print(k, list(v))

0 [np.int64(8), np.int64(4), np.int64(6)]
1 [np.int64(9), np.int64(5), np.int64(7), np.int64(3), np.int64(7)]


### **aggregate( ) 메서드**

In [20]:
# reduce(), fold(): 입출력 RDD 형식이 같지 않으면 오류가 발생
sc.parallelize([1, 2, 3, 4]).reduce(lambda x, y: (x + y, x * y)) 
    # single value -> pair 생성 시도할 경우 오류

IndentationError: unexpected indent (2377192267.py, line 3)

In [25]:
sc.parallelize([1, 2, 3, 4]).fold((0, 0), lambda x, y: (x[0] + y, x[1] + 1))

TypeError: unsupported operand type(s) for +: 'int' and 'tuple'

In [37]:
"""
    aggregate(): 입출력 형식 다른 RDD 에 대한 reduction을 수행
    - zeroValues: 파티션 초기 값
    - seqOp: 각 파티션에 대한 reduction 함수
    - combop: 파티션 결과를 합치는 reduction 함수
""" 

seqOp = (lambda x, y: (x[0] + y, x[1] + 1))
combOp = (lambda x, y: (x[0] + y[0], x[1] + y[1]))

In [38]:
# aggregate() 수행
sc.parallelize([1, 2, 3, 4], 2).aggregate((0,0), seqOp, combOp)

(10, 4)

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 42568)
Traceback (most recent call last):
  File "/usr/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/opt/spark/python/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/opt/spark/python/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
  File "/opt/spark/python/pyspark/accumulators.py", line 271, in accum_updates
    num_updates = read_int(self.rfile)
  File "/opt/spark/python/pyspark/serializers.py", lin